# Daily extrapolation

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt

import numpy as np
import rasterio as rio
import rasterio.warp
import xarray as xr
from pyproj import CRS
from sensorsio.utils import bb_transform

In [ ]:
from evaspa.common import daily

Input

In [ ]:
def setup_dataset(
    variables: list[str],
    date: dt.datetime,
    min_value: float = 0.0,
    max_value: float = 1.0,
    size: tuple[int, int] = (100, 200),
    epsg: int = 4326,
    wgs84_bounds: tuple[float, float, float, float] = (1.0, 43.0, 2.0, 44.0),
    seed: int = 0,
) -> xr.Dataset:
    """
    Create a rondom dataset
    """
    if len(variables) == 0:
        msg = "No variables"
        raise ValueError(msg)
    np.random.seed(seed)
    # Coordinate generation
    crs = CRS(epsg)
    bounds = bb_transform(
        str(CRS(4326)), str(crs), rio.coords.BoundingBox(*wgs84_bounds)
    )
    # BoundingBox(xmin, ymin, xmax, ymax)
    x = np.linspace(
        start=bounds.left,
        stop=bounds.right,
        num=size[1],
        dtype=np.float32,
    )
    y = np.linspace(
        start=bounds.bottom,
        stop=bounds.top,
        num=size[0],
        dtype=np.float32,
    )
    # Data generation
    data_vars = {}
    for var in variables:
        data_vars[var] = (
            ["y", "x"],
            np.random.uniform(low=min_value, high=max_value, size=size),
        )
    return xr.Dataset(
        data_vars=data_vars,
        coords={
            "y": ("y", y),
            "x": ("x", x),
        },
        attrs={"description": "Test data", "crs": crs, "date": date},
    )

In [ ]:
data = setup_dataset(
    ["var"],
    dt.datetime(2025, 1, 9, 11, 30, 00, tzinfo=dt.timezone.utc),
    epsg=32631,
    wgs84_bounds=(1.0, 43.0, 2.0, 44.0),
    size=(10, 20),
    max_value=105,
    min_value=95,
)
data

In [ ]:
data.sizes

In [ ]:
data["var"].plot()

In [ ]:
daily_xr = daily.extrapolate_at_daily_scale(data, method="toa")

In [ ]:
daily_xr

In [ ]:
daily_xr["var"].plot()